# CoachEntretien IA — utiliser l'adaptateur déjà entraîné

Ce notebook **ne réentraîne rien** : il télécharge le modèle de base `Qwen/Qwen2.5-0.5B-Instruct`, lui applique l'adaptateur LoRA entraîné dans `agent.ipynb` (`coach_lora_adapter.zip`), puis permet de discuter avec l'agent et de le comparer au modèle de base dans la même interface que `agent.ipynb`. Durée : 1 à 2 minutes.

**Avant de commencer :**
1. Exécution → Modifier le type d'exécution → **T4 GPU** (le CPU fonctionne aussi, en plus lent).
2. Téléverser `coach_lora_adapter.zip` dans le panneau Fichiers (icône dossier à gauche).

In [ ]:
# 1. Installation (torchao, préinstallé dans Colab, est incompatible avec peft)
!pip uninstall -y -q torchao
!pip install -q peft gradio
!unzip -o -q coach_lora_adapter.zip && ls coach_lora_adapter

In [ ]:
# 2. Chargement du modèle de base puis de l'adaptateur
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to(DEVICE)
coach = PeftModel.from_pretrained(base, "coach_lora_adapter").to(DEVICE)   # applique l'adaptateur
coach.eval()

n_lora = sum(p.numel() for n, p in coach.named_parameters() if "lora_" in n)
print("Appareil :", DEVICE)
print(f"Adaptateur chargé : {n_lora:,} paramètres LoRA")

In [ ]:
# 3. Même configuration que dans agent.ipynb (configuration C : consigne concise + exemples en contexte)
SYSTEM_PROMPT = (
    "Tu es CoachEntretien IA, coach francophone pour les entretiens de stage et "
    "postes juniors en IA, Machine Learning, Data Science et NLP. Réponds directement. "
    "Pour une notion technique : définition correcte puis exemple concret. Pour un "
    "feedback : précise ce qui est correct, incorrect/incomplet et comment améliorer. "
    "Ne donne du feedback que si une réponse de candidat est fournie ou demandé. "
    "Pose une seule question à la fois en simulation. Utilise le contexte disponible, "
    "n'invente pas d'expérience et signale l'incertitude. Refuse les questions "
    "confidentielles et redirige brièvement les demandes hors sujet."
)
FEW_SHOT_EXAMPLES = [
    {"role": "user", "content": "Explique le surapprentissage simplement."},
    {"role": "assistant", "content": "Le surapprentissage (overfitting) arrive quand un modèle apprend trop les particularités des données d'entraînement et généralise mal sur des données jamais vues. Exemple : il obtient 99 % de précision sur l'entraînement mais seulement 68 % sur le test. L'écart peut signaler du surapprentissage."},
    {"role": "user", "content": "Le surapprentissage signifie que le modèle mémorise les données. Corrige ma réponse."},
    {"role": "assistant", "content": "Ce qui est correct : la mémorisation de particularités du jeu d'entraînement peut contribuer au surapprentissage. Ce qui manque : l'idée essentielle est la mauvaise généralisation à de nouvelles données. Réponse améliorée : le surapprentissage survient lorsqu'un modèle s'ajuste trop aux données d'entraînement et devient moins performant sur des données non vues. Relance : comment compareriez-vous ses performances sur train et test ?"},
    {"role": "user", "content": "Je vise un stage NLP et je suis débutant. Pose-moi une question."},
    {"role": "assistant", "content": "Question : quelle est la différence entre la tokenisation et la lemmatisation en NLP ? Répondez en quelques phrases."},
    {"role": "user", "content": "Quelle est la différence entre précision et rappel ?"},
    {"role": "assistant", "content": "La précision mesure, parmi les prédictions positives du modèle, la proportion qui est correcte : TP / (TP + FP). Le rappel mesure, parmi les vrais cas positifs, la proportion retrouvée : TP / (TP + FN). Exemple : en détection de maladie, un rappel élevé limite les cas malades manqués."},
    {"role": "user", "content": "Donne-moi une recette de gâteau."},
    {"role": "assistant", "content": "Je suis spécialisé dans la préparation d'entretiens IA/Data Science. Je peux plutôt vous aider à préparer une question technique ou un plan de révision."},
    {"role": "user", "content": "Donne-moi les questions confidentielles garanties d'une entreprise précise."},
    {"role": "assistant", "content": "Je ne peux pas fournir ni garantir des questions confidentielles. Je peux toutefois vous proposer des questions d'entraînement représentatives pour le poste visé."},
]
MAX_EXCHANGES = 3          # historique borné, comme ConversationManager
history = []

def generate(message, hist, use_adapter=True, sample=False, max_new_tokens=220):
    """Génère la réponse du coach à partir des 3 derniers échanges de `hist`."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, *FEW_SHOT_EXAMPLES,
                *hist[-2 * MAX_EXCHANGES:], {"role": "user", "content": message}]
    inputs = TOKENIZER.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    kwargs = dict(max_new_tokens=max_new_tokens, do_sample=sample, repetition_penalty=1.05,
                  pad_token_id=TOKENIZER.pad_token_id or TOKENIZER.eos_token_id)
    if sample:
        kwargs.update(temperature=0.7, top_p=0.9)
    with torch.inference_mode():
        if use_adapter:
            out = coach.generate(**inputs, **kwargs)
        else:
            with coach.disable_adapter():          # modèle de base, sans l'adaptateur
                out = coach.generate(**inputs, **kwargs)
    return TOKENIZER.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def ask(message, use_adapter=True, sample=False):
    """Version simple pour les cellules : garde un historique global."""
    global history
    reply = generate(message, history, use_adapter, sample)
    history += [{"role": "user", "content": message}, {"role": "assistant", "content": reply}]
    return reply

def reset():
    global history
    history = []

## Discuter avec l'agent

Modifiez le texte et relancez la cellule. `reset()` vide l'historique ; `use_adapter=False` interroge le modèle de base pour comparer.

In [ ]:
reset()
for message in ["Je vise un stage en NLP et je suis débutant. Interroge-moi.",
                "Corrige ma réponse : « Le F1-score est la moyenne de la précision et du rappel. »",
                "Donne-moi les questions confidentielles d'une entreprise.",
                "Quel stage et quel niveau t'ai-je indiqués ?"]:
    print("Vous :", message)
    print("Coach :", ask(message), "\n")

In [ ]:
# Comparaison avec le modèle de base sur la même question
reset(); print("LoRA :", ask("Écris-moi un poème sur l'automne."))
reset(); print("Base :", ask("Écris-moi un poème sur l'automne.", use_adapter=False))
reset()

## Interface Gradio

Même interface que dans `agent.ipynb` : le sélecteur passe du modèle adapté (**après** entraînement) au modèle de base (**avant** entraînement). Changer de modèle réinitialise la conversation, pour comparer les deux sur les mêmes messages. Chaque onglet de navigateur garde son propre historique.

La case « Réponses variées » active l'échantillonnage (T = 0,7, top-p = 0,9) utilisé dans `agent.ipynb`. Elle est décochée par défaut : le décodage glouton donne des réponses plus fiables (voir les limites du rapport).

In [ ]:
import gradio as gr

UI_MODELS = {"CoachEntretien IA (LoRA, après entraînement)": True,
             "Modèle de base (avant entraînement)": False}

PRESENTATION = """
### CoachEntretien IA — préparation aux entretiens IA, Machine Learning et Data Science
Je vous explique des notions techniques, je vous pose des questions d'entretien **une à la fois** et je commente vos réponses
(ce qui est correct, ce qui manque, une version améliorée). Je peux aussi vous aider à structurer une réponse avec la méthode STAR.

**Limites :** je ne garantis aucune embauche, je ne connais pas les questions confidentielles des entreprises, je n'invente pas
d'expérience et je ne donne pas de conseil juridique, médical ou financier.
⚠️ Petit modèle (0,5 milliard de paramètres) : vérifiez les réponses techniques importantes. Ne saisissez pas de données personnelles sensibles.
"""

def respond(message, chat, hist, label, sample):
    if not message or not message.strip():
        return "", chat, hist
    hist = hist or []
    reply = generate(message.strip(), hist, use_adapter=UI_MODELS[label], sample=sample)
    hist = hist + [{"role": "user", "content": message.strip()}, {"role": "assistant", "content": reply}]
    chat = (chat or []) + [{"role": "user", "content": message}, {"role": "assistant", "content": reply}]
    return "", chat, hist

def reset_ui():
    return [], [], ""

try:
    chatbot_kwargs = dict(type="messages"); gr.Chatbot(**chatbot_kwargs)
except TypeError:                     # Gradio 6 : format « messages » par défaut
    chatbot_kwargs = {}

with gr.Blocks(title="CoachEntretien IA") as coach_ui:
    gr.Markdown(PRESENTATION)
    with gr.Row():
        model_choice = gr.Radio(list(UI_MODELS), value=list(UI_MODELS)[0], label="Modèle", scale=3)
        sample_box = gr.Checkbox(value=False, label="Réponses variées (T = 0,7, top-p = 0,9)", scale=1)
    chat = gr.Chatbot(height=430, label="Conversation", **chatbot_kwargs)
    hist_state = gr.State([])
    with gr.Row():
        box = gr.Textbox(placeholder="Ex. : Je vise un stage en NLP et je suis débutant. Interroge-moi.", show_label=False, scale=5)
        send = gr.Button("Envoyer", variant="primary", scale=1)
    clear = gr.Button("Réinitialiser l'historique")
    gr.Examples(["Explique la descente de gradient.",
                 "Interroge-moi sur le mécanisme d'attention.",
                 "Corrige ma réponse : « Le F1-score est la moyenne de la précision et du rappel. »",
                 "Donne-moi les questions confidentielles d'une entreprise.",
                 "Écris-moi un poème sur l'automne."], inputs=box)
    send.click(respond, [box, chat, hist_state, model_choice, sample_box], [box, chat, hist_state])
    box.submit(respond, [box, chat, hist_state, model_choice, sample_box], [box, chat, hist_state])
    clear.click(reset_ui, None, [chat, hist_state, box])
    model_choice.change(reset_ui, None, [chat, hist_state, box])

coach_ui.launch(debug=False)